# 📘 Annotated edition · A.Data_Loading

A copy of `notebooks/A.Data_Loading.ipynb`. Original cells and saved outputs are unchanged. New **📘 Explanation** cells are added. This notebook asks no explicit questions; the ✅ notes at the end cover the implicit ones ("is it safe to run, and how does it relate to 02?").

**What this notebook is:** a **standalone, self-contained loader** for the bookings folder. It does the same job as `loaders.read_legacy_file` / `read_nova_file` + the concat in notebook 02, but without importing the `kalpa` package and without a process pool. It also handles **RAR archives** in case the data pack arrives as `.rar` files.

**Why its numbers differ from notebook 02:** the saved output (58,362 rows, 25,069 bookings from 5 files) was produced on an earlier **small sample** of the pack. The folder now holds the full pack (108 booking files), so a re-run would read about 1.3 million rows.

⚠️ **It writes `data/clean/bookings_all.csv.gz`, the same file notebook 02 writes.** Running it after 02 replaces 02's output. Use a different file name, or run only one of the two.

> ℹ️ **One line changed in this annotated copy:** `ROOT = ... if pathlib.Path.cwd().name in ("notebooks", "notebook_new") else ...`. The original only recognised a folder called `notebooks`. Run from `notebook_new/`, it would have looked for `notebook_new/data/raw` and failed. Nothing else in the code was changed.

# A · Load bookings from `data/raw/bookings`

Legacy CSV is read as Latin-1 at booking-test grain. Nova JSONL is read as UTF-8, nested tests are flattened, and timestamps are converted to Asia/Kolkata. The supplied sample is gzip-compressed; RAR extraction is supported when RAR files are present.

#### 📘 Explanation: header

- **Legacy CSV** is read as **Latin-1**, at booking-test grain (one row per test).
- **Nova JSONL** is read as **UTF-8**. Nested tests are **flattened** to one row per test. Timestamps (which carry a timezone) are converted to **Asia/Kolkata** (IST) and the timezone label is dropped, so they compare with legacy's local times.
- Sample files are `.gz`. If `.rar` files are present, they're extracted first.

#### 📘 Explanation: setup and file discovery

- `from __future__ import annotations`: allows modern type hints on older Python versions (harmless here).
- `gzip, io, json, pathlib, tempfile`: decompression, in-memory text buffers, JSON parsing, paths, temporary folders.
- `ROOT`, `BOOKINGS_ROOT = ROOT/'data'/'raw'/'bookings'`, `CLEAN_ROOT` (created if missing).
- `LEGACY_DTYPES`: the same declared dtypes as `loaders.py` (ids as `string`, low-cardinality fields as `category`, prices as nullable `Int32`).
- `discover(root)` walks every file under the bookings folder and keeps `.rar`, `.gz`, `.csv` and `.jsonl` files, tagging each one:
  - `archive` if `.rar`,
  - `nova` if the name contains `.jsonl`,
  - otherwise `legacy`.
  It returns a DataFrame of `path`, `kind`, `bytes`.
- The last line displays the table with paths shown relative to the project root (`files.assign(path=...)` returns a modified copy and leaves `files` unchanged).

In [1]:
from __future__ import annotations
import gzip, io, json, pathlib, tempfile
import pandas as pd
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name in ('notebooks', 'notebook_new') else pathlib.Path.cwd()
BOOKINGS_ROOT = ROOT / 'data' / 'raw' / 'bookings'
CLEAN_ROOT = ROOT / 'data' / 'clean'
CLEAN_ROOT.mkdir(parents=True, exist_ok=True)
LEGACY_DTYPES = {'booking_id': 'string', 'city': 'category', 'clinic_id': 'category', 'channel': 'category', 'source': 'category', 'patient_id': 'string', 'patient_name': 'string', 'test_code': 'category', 'test_price': 'Int32', 'home_fee': 'Int32'}

def discover(root):
    rows = []
    for p in sorted(root.rglob('*')):
        if p.is_file() and p.suffix.lower() in {'.rar', '.gz', '.csv', '.jsonl'}:
            kind = 'archive' if p.suffix.lower() == '.rar' else ('nova' if '.jsonl' in p.name.lower() else 'legacy')
            rows.append({'path': p, 'kind': kind, 'bytes': p.stat().st_size})
    return pd.DataFrame(rows)
files = discover(BOOKINGS_ROOT)
files.assign(path=files.path.map(lambda p: p.relative_to(ROOT)))

,path,kind,bytes
0,data\raw\bookings\legacy\bengaluru\bookings_20...,legacy,215062
1,data\raw\bookings\legacy\chennai\bookings_2026...,legacy,158526
2,data\raw\bookings\legacy\jaipur\bookings_2026-...,legacy,80601
3,data\raw\bookings\nova\chennai\bookings_2026-0...,nova,145555
4,data\raw\bookings\nova\jaipur\bookings_2026-04...,nova,71895


#### 📘 Reading the output
In the saved run, 5 files: 3 legacy (Bengaluru, Chennai, Jaipur) and 2 nova (Chennai, Jaipur), the small sample of the time. Today `discover` would list 108 files (102 legacy + 6 nova).

#### 📘 Explanation: RAR handling and the two readers

**RAR extraction:**
- `tempfile.TemporaryDirectory(prefix='kalpa-bookings-')`: a temporary folder that's deleted automatically when the object is cleaned up.
- If there are `.rar` files, the `rarfile` library is imported **lazily** (`importlib.import_module`), so it's only needed when RARs exist. If it's missing, a clear `RuntimeError` says what to install. `rarfile` also needs an external `unrar`/`bsdtar` backend.
- Each archive is extracted into its own subfolder, and the extracted files join `input_paths`.
- With no RARs (as now), `input_paths` is just the non-archive files.

**`text_handle(path, encoding)`** opens `.gz` files with `gzip.open(..., 'rt')` (decompress + decode) and plain files with `path.open('rt')`, so the readers work on extracted, uncompressed files too.

**`read_legacy(path)`**, the same logic as `loaders.read_legacy_file`:
1. Read all lines as Latin-1.
2. `header = lines[0]`. Drop every later line identical to the header (append artefacts).
3. Parse with `pd.read_csv(io.StringIO(...), dtype=LEGACY_DTYPES)`.
4. Dates with an explicit day-first format `'%d/%m/%Y %H:%M'`, `errors='coerce'`.
5. Strip `booking_id`, and add `source_file` and `system='legacy'`.
6. Store the repeated-header count in `frame.attrs`.

**`read_nova(path)`**, the same logic as `loaders.read_nova_file`:
1. For each JSON line (one booking), for each test in `record['tests']`, append one flat row with the booking's fields, the test's `code` and `price`, and `patient_id` from the nested `record['patient']`. `channel` and `source` are lower-cased. `patient_name` is NA (nova doesn't export it). `rescheduled_from` and `status_at_export` use `.get()` because not every record has them.
2. `pd.to_datetime(..., utc=True).dt.tz_convert('Asia/Kolkata').dt.tz_localize(None)`: parse as UTC-aware, convert to IST, then drop the zone so values are plain IST timestamps like legacy's.

In [2]:
rar_tmp = tempfile.TemporaryDirectory(prefix='kalpa-bookings-')
rar_files = files.loc[files.kind.eq('archive'), 'path'].tolist() if not files.empty else []
extracted = []
if rar_files:
    try:
        import importlib
        rarfile = importlib.import_module('rarfile')
    except ImportError as exc:
        raise RuntimeError('RAR input found: install rarfile and a RAR backend before loading.') from exc
    for archive_path in rar_files:
        target = pathlib.Path(rar_tmp.name) / archive_path.stem
        target.mkdir(parents=True, exist_ok=True)
        with rarfile.RarFile(archive_path) as archive:
            archive.extractall(target)
        extracted.extend(p for p in target.rglob('*') if p.is_file())
input_paths = files.loc[files.kind.ne('archive'), 'path'].tolist() + extracted

def text_handle(path, encoding):
    return gzip.open(path, 'rt', encoding=encoding) if path.name.lower().endswith('.gz') else path.open('rt', encoding=encoding)

def read_legacy(path):
    with text_handle(path, 'latin-1') as handle:
        lines = handle.read().splitlines()
    header = lines[0]
    body = [line for line in lines[1:] if line != header]
    frame = pd.read_csv(io.StringIO('\n'.join([header, *body])), dtype=LEGACY_DTYPES)
    for column in ('created_at', 'slot_at'):
        frame[column] = pd.to_datetime(frame[column], format='%d/%m/%Y %H:%M', errors='coerce')
    frame['booking_id'] = frame['booking_id'].str.strip()
    frame['source_file'], frame['system'] = path.name, 'legacy'
    frame.attrs['repeated_header_lines'] = len(lines) - 1 - len(body)
    return frame

def read_nova(path):
    rows = []
    with text_handle(path, 'utf-8') as handle:
        for line in handle:
            record = json.loads(line)
            for test in record['tests']:
                rows.append({'booking_id': record['booking_id'], 'created_at': record['created_at'], 'slot_at': record['slot_at'], 'city': record['city'], 'clinic_id': record['clinic_id'], 'channel': record['channel'].lower(), 'source': record['source'].lower(), 'patient_id': record['patient']['patient_id'], 'patient_name': pd.NA, 'test_code': test['code'], 'test_price': test['price'], 'home_fee': record['home_fee'], 'rescheduled_from': record.get('rescheduled_from'), 'status_at_export': record.get('status_at_export')})
    frame = pd.DataFrame(rows)
    for column in ('created_at', 'slot_at'):
        frame[column] = pd.to_datetime(frame[column], utc=True).dt.tz_convert('Asia/Kolkata').dt.tz_localize(None)
    frame['source_file'], frame['system'] = path.name, 'nova'
    return frame

#### 📘 Explanation: read everything, align columns, concatenate, check, save

1. Loop over `input_paths` in sorted order and dispatch by name: `.jsonl` to `read_nova`, `.csv` to `read_legacy`, anything else raises `ValueError` (an unexpected file type is an error, not something to skip silently).
2. **Column union in first-seen order:** `columns` collects every column name from every frame, keeping the order of first appearance. Then each frame gets any missing column added as `pd.NA`, and `pd.concat([frame[columns] ...])` stacks them in one consistent column order.
3. `Bangalore` → `Bengaluru`.
4. Sum the repeated-header counts.
5. **Two hard `assert`s** (unlike `checks.check`, these **stop** the notebook if they fail): the result isn't empty and every `booking_id` is present, and every `created_at` and `slot_at` parsed (no `NaT`).
6. Save as gzip CSV and print a PASS line and the output path, then `head()`.

In [3]:
frames = []
for path in sorted(input_paths, key=str):
    if '.jsonl' in path.name.lower():
        frames.append(read_nova(path))
    elif '.csv' in path.name.lower():
        frames.append(read_legacy(path))
    else:
        raise ValueError(f'Unsupported extracted booking file: {path}')
columns = []
for frame in frames:
    columns.extend(column for column in frame.columns if column not in columns)
for frame in frames:
    for column in set(columns).difference(frame.columns):
        frame[column] = pd.NA
bookings_all = pd.concat([frame[columns] for frame in frames], ignore_index=True)
bookings_all['city'] = bookings_all['city'].astype('string').replace({'Bangalore': 'Bengaluru'})
repeated_headers = sum(frame.attrs.get('repeated_header_lines', 0) for frame in frames)
assert not bookings_all.empty and bookings_all['booking_id'].notna().all()
assert bookings_all['created_at'].notna().all() and bookings_all['slot_at'].notna().all()
output_path = CLEAN_ROOT / 'bookings_all.csv.gz'
bookings_all.to_csv(output_path, index=False, compression='gzip')
print(f'PASS: {len(bookings_all):,} booking-test rows; {bookings_all.booking_id.nunique():,} bookings; {repeated_headers} repeated headers dropped')
print(f'wrote {output_path}')
bookings_all.head()

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_17900\3988241335.py:15: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  bookings_all = pd.concat([frame[columns] for frame in frames], ignore_index=True)


PASS: 58,362 booking-test rows; 25,069 bookings; 0 repeated headers dropped
wrote c:\Users\Lenovo\Desktop\Pure_Kalpa_Data_Folder\data\clean\bookings_all.csv.gz


,booking_id,created_at,slot_at,city,clinic_id,channel,source,patient_id,patient_name,test_code,test_price,home_fee,source_file,system,rescheduled_from,status_at_export
0,BK-BLR-202503-014680,2025-03-21 03:40:00,2025-03-23 07:00:00,Bengaluru,BLR-03,clinic,app,PBLR0009350,Diya Shah,T005,700,0,bookings_2025-03.csv.gz,legacy,NaN,NaN
1,BK-BLR-202503-014681,2025-03-03 09:58:00,2025-03-04 16:51:00,Bengaluru,BLR-01,clinic,walkin,PBLR0006384,Ritika Gupta,T008,1100,0,bookings_2025-03.csv.gz,legacy,NaN,NaN
2,BK-BLR-202503-014682,2025-03-21 04:34:00,2025-03-21 07:44:00,Bengaluru,BLR-01,home,phone,PBLR0029923,Ishaan Nair,T001,350,150,bookings_2025-03.csv.gz,legacy,NaN,NaN
3,BK-BLR-202503-014682,2025-03-21 04:34:00,2025-03-21 07:44:00,Bengaluru,BLR-01,home,phone,PBLR0029923,Ishaan Nair,T008,1100,150,bookings_2025-03.csv.gz,legacy,NaN,NaN
4,BK-BLR-202503-014682,2025-03-21 04:34:00,2025-03-21 07:44:00,Bengaluru,BLR-01,home,phone,PBLR0029923,Ishaan Nair,T005,700,150,bookings_2025-03.csv.gz,legacy,NaN,NaN


#### 📘 Reading the output

- **FutureWarning** about concatenating all-NA columns: the same harmless warning as in notebook 02, caused by the `pd.NA` columns added to legacy frames.
- **PASS: 58,362 booking-test rows; 25,069 bookings; 0 repeated headers dropped**, for the 5-file sample of that time.
- `head()` shows `BK-BLR-202503-014682` on three rows, meaning one booking with three tests (booking-test grain).

#### 📘 Explanation
`bookings_all.info()` prints each column's non-null count and dtype, plus memory use. It's the quickest way to spot missing values and wrong types.

In [4]:
bookings_all.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 58362 entries, 0 to 58361
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   booking_id        58362 non-null  object        
 1   created_at        58362 non-null  datetime64[ns]
 2   slot_at           58362 non-null  datetime64[ns]
 3   city              58362 non-null  string        
 4   clinic_id         58362 non-null  object        
 5   channel           58362 non-null  object        
 6   source            58362 non-null  object        
 7   patient_id        58362 non-null  object        
 8   patient_name      37658 non-null  string        
 9   test_code         58334 non-null  object        
 10  test_price        58362 non-null  Int64         
 11  home_fee          58362 non-null  Int64         
 12  source_file       58362 non-null  object        
 13  system            58362 non-null  object        
 14  rescheduled_from  1169

#### 📘 Reading the output
16 columns, 58,362 rows. All key columns are fully populated. Note the dtypes: `booking_id`, `clinic_id`, `channel` and `source` became **`object`**, not `string` or `category`, because concatenating frames whose category sets differ (and mixing legacy `string` with nova `object`) **falls back to `object`**. That's the concat trap described in notebook 01's interview angle. If memory matters, re-apply `astype("category")` after concatenating.

#### 📘 Explanation
An empty cell, left at the end of the original notebook. It does nothing.

---
#### ✅ Notes on the implicit questions

**How does this relate to notebook 02?** Same readers and same output grain, but no process pool, no pydantic validation and no booking-grain roll-up. Use 02 for the project pipeline (it validates and writes the rejects file). This notebook is useful as a dependency-free loader, or when the pack arrives as RAR files.

**Is it safe to run?** It **overwrites `data/clean/bookings_all.csv.gz`**. If you run it after 02, 02's version is replaced. The content would be the same rows, but the column order and dtypes can differ. Safer: change `output_path` to `CLEAN_ROOT / 'bookings_all_A.csv.gz'`.

**Would a re-run give the same numbers as 02?** On the full pack, yes for the counts: 1,315,183 booking-test rows and 567,986 bookings, with 21 repeated headers dropped, since the reading rules are identical.